# Notebook 00 — Underwater Dataset Preparation (SINTEF LIACI → YOLO)

**Paper artifact:** Dataset 2 description — *"Underwater corrosion dataset (SINTEF LIACI)"* (Section 4.1.2).

This notebook curates the raw **SINTEF LIACI** underwater inspection imagery into a clean, YOLO-formatted detection dataset for the corrosion class. It consolidates the logic of two standalone scripts — `cambio.py` (raw → YOLO base) and `convert_masks.py` (segmentation polygons → bounding boxes) — into a single, reproducible pipeline.

The procedure (i) inspects every segmentation mask to separate **genuinely positive** images (non-empty corrosion mask) from **genuinely negative** ones (all-black mask or no mask), (ii) resizes images and masks to a fixed input size and emits YOLO segmentation labels, (iii) injects a controlled fraction of corrosion-free *negatives* to expose the detector to true background, and (iv) optionally derives axis-aligned bounding-box labels from the segmentation polygons.

**Key result.** The pipeline yields the curated underwater set used throughout the paper: **N = 261** images — **209** positives (with corrosion) and **52** negatives (background). This is the dataset behind the paper's Dataset 2.

---

| | |
|---|---|
| **Inputs** | raw SINTEF LIACI tree under `./dataset(acuatico)/` (`images/`, `masks/corrosion/`, `train_test_split.csv`) |
| **Output** | YOLO dataset in `./dataset_yolo/` (`images/{train,test}`, `labels/{train,test}`, `dataset.yml`) → paper Dataset 2 (Section 4.1.2) |
| **Environment** | Python 3 · Pillow · OpenCV · NumPy · pandas |

> **Note on outputs.** All code cells and their outputs are preserved exactly as executed on the original run; console messages appear in their original form (including the original Windows-style path separators printed by the run). Re-running the pipeline reproduces the same dataset counts.


## 1 · Configuration and imports

Load the image/array stack (Pillow, OpenCV, NumPy, pandas) and declare the global configuration: the raw-dataset root, the YOLO output root, the fixed target image size, the negatives ratio, and all derived input/output paths. Centralising these constants keeps the pipeline reproducible and easy to re-point at a different dataset location.


In [1]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import random
import cv2

# =============================
# GLOBAL CONFIGURATION
# =============================
DATASET_ROOT = "./dataset(acuatico)"
OUTPUT_ROOT = "./dataset_yolo"
TARGET_SIZE = (640, 640)

# Fraction of corrosion-free images to retain (to control false positives)
NEGATIVES_RATIO = 0.25
# Target class of interest
TARGET_CLASS = "corrosion"

# Derived paths
IMAGES_DIR = os.path.join(DATASET_ROOT, "images")
MASKS_DIR = os.path.join(DATASET_ROOT, "masks")
SPLIT_CSV = os.path.join(DATASET_ROOT, "train_test_split.csv")

# YOLO output paths
YOLO_IMAGES_TRAIN = os.path.join(OUTPUT_ROOT, "images", "train")
YOLO_IMAGES_TEST  = os.path.join(OUTPUT_ROOT, "images", "test")
YOLO_LABELS_TRAIN = os.path.join(OUTPUT_ROOT, "labels", "train")
YOLO_LABELS_TEST  = os.path.join(OUTPUT_ROOT, "labels", "test")

# Bounding-box output paths (for Part 2)
YOLO_LABELS_BBOX_TRAIN = os.path.join(OUTPUT_ROOT, "labels_bbox", "train")
YOLO_LABELS_BBOX_TEST = os.path.join(OUTPUT_ROOT, "labels_bbox", "test")

## 2 · Build the base dataset (logic of `cambio.py`)

This section creates the output directory tree, writes the YOLO `dataset.yml`, and then performs the critical **content-aware labelling step**: rather than trusting file names, every mask is opened and inspected. An image is classified as a *true positive* only if its corrosion mask contains at least one non-zero pixel; images with an all-black mask, an unreadable mask, or no mask at all are recorded as *true negatives*. This guards against mislabelled or empty masks contaminating the positive set.


In [2]:
# Create output directories
for folder in [YOLO_IMAGES_TRAIN, YOLO_IMAGES_TEST, YOLO_LABELS_TRAIN, YOLO_LABELS_TEST]:
    os.makedirs(folder, exist_ok=True)

# Write dataset.yml
dataset_yml_content = f"""
train: ./images/train
val: ./images/test
test: ./images/test

nc: 1
names: ['corrosion']
"""
with open(os.path.join(OUTPUT_ROOT, 'dataset.yml'), 'w') as f:
    f.write(dataset_yml_content)
print(f"✅ Created {os.path.join(OUTPUT_ROOT, 'dataset.yml')}")

# =============================
# IDENTIFY GENUINE POSITIVE AND NEGATIVE IMAGES
# =============================
print("🔍 Identifying genuinely positive and negative images (inspecting mask content)..." )

true_positive_basenames = set()
true_negative_info = {} # Stores basename: full_filename for negatives

target_class_path = os.path.join(MASKS_DIR, TARGET_CLASS)
all_image_files_in_dir = os.listdir(IMAGES_DIR)
all_images_info = {os.path.splitext(f)[0]: f for f in all_image_files_in_dir}

if not os.path.isdir(target_class_path):
    print(f"❌ Error: the mask directory for '{TARGET_CLASS}' does not exist: {target_class_path}")
    # If the mask directory does not exist, every image is effectively negative
    true_negative_info = all_images_info.copy()
else:
    for image_file in all_image_files_in_dir:
        base_name = os.path.splitext(image_file)[0]
        mask_file_path = None
        
        # Look up the corresponding mask file
        for f in os.listdir(target_class_path):
            if os.path.splitext(f)[0] == base_name:
                mask_file_path = os.path.join(target_class_path, f)
                break
        
        if mask_file_path:
            try:
                with Image.open(mask_file_path) as mask_img:
                    mask_array = np.array(mask_img.convert("L")) # Convert to grayscale
                    if np.sum(mask_array) == 0: # Check whether every pixel is black
                        true_negative_info[base_name] = image_file
                    else:
                        true_positive_basenames.add(base_name)
            except Exception as e:
                print(f"⚠️  Warning: could not process mask {mask_file_path}: {e}. Assuming it is negative.")
                true_negative_info[base_name] = image_file
        else:
            # If no mask file exists, it is also a true negative
            true_negative_info[base_name] = image_file

print(f"  - Found {len(true_positive_basenames)} genuinely positive images.")
print(f"  - Found {len(true_negative_info)} genuinely negative images.")

✅ Created ./dataset_yolo\dataset.yml
🔍 Identifying genuinely positive and negative images (inspecting mask content)...
  - Found 209 genuinely positive images.
  - Found 1684 genuinely negative images.


The content-aware scan separates the raw pool into **209** genuinely positive images (non-empty corrosion masks) and **1684** genuinely negative ones. Only the positives are processed in full below; a small, controlled subset of the negatives is added afterwards (Section 2, negatives step) so the detector still sees true background without being overwhelmed by it.


### 2.1 · Read the train/test split

Load the official `train_test_split.csv` and build a fast `basename → split` lookup. Any image absent from the CSV later falls back to the `train` partition. The positive/negative counters are initialised here.


In [3]:
# =============================
# READ SPLIT AND PREPARE LOOKUP
# =============================
split_df = pd.read_csv(SPLIT_CSV)
# Build a dictionary for fast split lookup: {basename: split}
split_lookup = {os.path.splitext(row['file_name'])[0]: row['split'].lower() for _, row in split_df.iterrows()}

positives_count = 0
negatives_count = 0

### 2.2 · Process the positive images

For every genuinely positive basename: resolve its split, resize the image to the target size with high-quality (LANCZOS) resampling, and convert its corrosion mask into YOLO **segmentation** labels. Each external contour with at least three vertices is normalised to `[0, 1]` coordinates and written as a `class x1 y1 … xn yn` polygon line.


In [4]:
# =============================
# PROCESS ALL POSITIVE IMAGES
# =============================
print(f"⚙️  Processing the {len(true_positive_basenames)} positive images found...")

for base_name in true_positive_basenames:
    # 1. Determine the split (train/test)
    split = split_lookup.get(base_name)

    # 2. If absent from the CSV, default to 'train'
    if split not in ["train", "test"]:
        split = "train"

    # 3. Retrieve the original file name
    image_name = all_images_info.get(base_name)
    if not image_name:
        print(f"⚠️  Warning: no file name found for basename {base_name}. Skipping.")
        continue

    positives_count += 1

    # Process image
    img_path = os.path.join(IMAGES_DIR, image_name)
    img_out_dir = YOLO_IMAGES_TRAIN if split == "train" else YOLO_IMAGES_TEST
    img_out_path = os.path.join(img_out_dir, image_name)

    try:
        with Image.open(img_path) as img:
            img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
            img_resized.save(img_out_path)
    except FileNotFoundError:
        print(f"⚠️  Warning: image not found {image_name}. Skipping.")
        positives_count -= 1
        continue

    # Process mask (polygons)
    mask_file = None
    for f in os.listdir(target_class_path):
        if os.path.splitext(f)[0] == base_name:
            mask_file = f
            break
    
    if not mask_file:
        continue

    lbl_out_dir = YOLO_LABELS_TRAIN if split == "train" else YOLO_LABELS_TEST
    label_lines = []

    mask_path = os.path.join(target_class_path, mask_file)
    with Image.open(mask_path) as mask:
        mask_gray = mask.convert("L")
        mask_resized = mask_gray.resize(TARGET_SIZE, Image.Resampling.NEAREST)
        mask_array = np.array(mask_resized)

        _, binary_mask = cv2.threshold(mask_array, 0, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            for contour in contours:
                if contour.shape[0] < 3:
                    continue
                
                normalized_contour = contour.squeeze(axis=1).astype(float)
                normalized_contour[:, 0] /= TARGET_SIZE[0]
                normalized_contour[:, 1] /= TARGET_SIZE[1]
                segment = normalized_contour.ravel().tolist()
                label_lines.append(f"0 {' '.join(f'{p:.6f}' for p in segment)}")

    if label_lines:
        label_path = os.path.join(lbl_out_dir, f"{base_name}.txt")
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))

⚙️  Processing the 209 positive images found...


### 2.3 · Inject negative (background) images

A deliberate fraction of corrosion-free images — `NEGATIVES_RATIO` of the positive count — is shuffled in as hard negatives. These images carry no label file (empty ground truth), teaching the detector what *absence* of corrosion looks like and reducing false positives at inference time.


In [5]:
# =============================
# PROCESS NEGATIVE IMAGES
# =============================
print("⚙️  Processing negative images...")

target_negatives = int(positives_count * NEGATIVES_RATIO)
print(f"  - Target: add {target_negatives} negative images ({NEGATIVES_RATIO:.0%} of the {positives_count} positives).")

shuffled_negative_basenames = list(true_negative_info.keys())
random.shuffle(shuffled_negative_basenames)

selected_negatives = shuffled_negative_basenames[:target_negatives]

for base_name in selected_negatives:
    negatives_count += 1
    split = "train" if random.random() < 0.8 else "test"
    
    image_name = true_negative_info[base_name]
    img_path = os.path.join(IMAGES_DIR, image_name)
    img_out_dir = YOLO_IMAGES_TRAIN if split == "train" else YOLO_IMAGES_TEST
    img_out_path = os.path.join(img_out_dir, image_name)

    try:
        with Image.open(img_path) as img:
            img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
            img_resized.save(img_out_path)
    except FileNotFoundError:
        print(f"⚠️  Warning: negative image not found {image_name}. Skipping.")
        negatives_count -= 1
        continue

print(f"✅ YOLO dataset successfully created at: {OUTPUT_ROOT}")
print(f"   - Images with corrosion (positives): {positives_count}")
print(f"   - Images without corrosion (negatives): {negatives_count}")

⚙️  Processing negative images...
  - Target: add 52 negative images (25% of the 209 positives).
✅ YOLO dataset successfully created at: ./dataset_yolo
   - Images with corrosion (positives): 209
   - Images without corrosion (negatives): 52


This is the headline dataset for the paper: **209** positive images (with corrosion) and **52** negative images (background), for **N = 261** total. The `25%` negatives ratio is the empirical balance reported for Dataset 2; the curated set is now written under `./dataset_yolo/` in standard YOLO layout and is ready for training and evaluation in the downstream notebooks.


## 3 · Convert masks to bounding boxes (logic of `convert_masks.py`)

This stage is **optional**. The training pipeline can consume the segmentation polygons directly; run this step only when axis-aligned **bounding-box** labels are required for pure object detection. For each label file it reads the segmentation polygons, takes the per-polygon coordinate extrema, and rewrites the file *in place* in YOLO detection format (`class x_center y_center width height`).


In [6]:
def convert_masks_to_bboxes_inplace(labels_dir):
    if not os.path.exists(labels_dir):
        print(f"Warning: the label directory does not exist: {labels_dir}")
        return

    print(f"🔄 Converting masks to bounding boxes IN-PLACE in {labels_dir}...")
    count = 0
    for filename in os.listdir(labels_dir):
        if filename.endswith(".txt"):
            filepath = os.path.join(labels_dir, filename)

            try:
                # Read the original content (segmentation)
                with open(filepath, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 3: continue
                    
                    class_id = parts[0]
                    # YOLO Segmentation: class x1 y1 x2 y2 ... xn yn
                    coords = np.array([float(p) for p in parts[1:]]).reshape(-1, 2)

                    x_min = np.min(coords[:, 0])
                    y_min = np.min(coords[:, 1])
                    x_max = np.max(coords[:, 0])
                    y_max = np.max(coords[:, 1])

                    x_center = (x_min + x_max) / 2
                    y_center = (y_min + y_max) / 2
                    width = x_max - x_min
                    height = y_max - y_min

                    # YOLO Detection: class x_center y_center width height
                    new_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
                
                # Overwrite the file with the new content (bounding box)
                if new_lines:
                    with open(filepath, 'w') as f:
                        f.write("\n".join(new_lines))
                    count += 1
            except Exception as e:
                print(f"❌ Error converting {filename}: {e}")
    
    print(f"  ✅ {count} files converted and updated to bounding-box format.")

In [7]:
# Run the IN-PLACE conversion for the train and test splits
# This directly modifies the files under dataset_yolo/labels/...
convert_masks_to_bboxes_inplace(YOLO_LABELS_TRAIN)
convert_masks_to_bboxes_inplace(YOLO_LABELS_TEST)

print("\n🚀 Conversion complete: the label files now contain bounding boxes.")

🔄 Converting masks to bounding boxes IN-PLACE in ./dataset_yolo\labels\train...
  ✅ 188 files converted and updated to bounding-box format.
🔄 Converting masks to bounding boxes IN-PLACE in ./dataset_yolo\labels\test...
  ✅ 21 files converted and updated to bounding-box format.

🚀 Conversion complete: the label files now contain bounding boxes.


The in-place conversion rewrites **188** training and **21** test label files from segmentation polygons to bounding boxes. After this cell, the files under `dataset_yolo/labels/` hold detection-format boxes; if both label types are needed, run the converter on a copy rather than in place.
